In [ ]:
# ==========================================================================================
# CELL 1: SETUP REPOSITORY & DEPENDENCIES
# ==========================================================================================
import os

if not os.path.exists('/kaggle/working/remote_sensing'):
    !git clone https://github.com/doantrongthai/remote_sensing.git /kaggle/working/remote_sensing
else:
    %cd /kaggle/working/remote_sensing
    !git pull

%cd /kaggle/working/remote_sensing
!pip install -q -r requirements.txt


In [ ]:
# ==========================================================================================
# CELL 2: VERIFY DUAL-GPU (T4 x 2) & DATASET
# ==========================================================================================
import os
import torch

data_dir = "/kaggle/input/datasets/doantrongthai/sen1floods11"
n_gpus = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {n_gpus}")
for i in range(n_gpus):
    print(f"  - GPU {i}: {torch.cuda.get_device_name(i)} (VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")
print(f"Dataset Path Exists: {os.path.exists(data_dir)}")
if os.path.exists(data_dir):
    print("Splits available:", os.listdir(data_dir))


In [ ]:
# ==========================================================================================
# CELL 3: RUN ALL 14 MODELS DUAL-GPU PARALLEL (METHOD A - OVERNIGHT SWEEP)
# GPU 0 (7 Models): shadowfree_net, unet, deeplabv3plus, hrnet, transunet, farseg, manet
# GPU 1 (7 Models): procanet, unetplusplus, pspnet, segformer, swin_unet, attentive_unet, pan
# Mỗi model tự động chạy trọn vẹn 5 Seeds: [1, 2, 3, 4, 5]
# Cả 2 GPU cày đồng thời 100% công suất -> Tiết kiệm 50% thời gian chạy đêm!
# ==========================================================================================
%cd /kaggle/working/remote_sensing

!python scripts/runner_dual_gpu.py \
    --data_dir /kaggle/input/datasets/doantrongthai/sen1floods11 \
    --all \
    --seeds 1 2 3 4 5 \
    --epochs 25 \
    --batch_size 16


In [ ]:
# ==========================================================================================
# CELL 4: BÁO CÁO TỔNG KẾT: MULTI-SEED GRAND BENCHMARK LEADERBOARD (Mean ± Std)
# ==========================================================================================
%cd /kaggle/working/remote_sensing
import json
import os
import glob

save_dir = './checkpoints'
res_file = os.path.join(save_dir, 'benchmark_results.json')

# Merge all per-model json files
merged = {}
for p in glob.glob(os.path.join(save_dir, '*_results.json')):
    if os.path.basename(p) == 'benchmark_results.json': continue
    try:
        with open(p, 'r') as f:
            d = json.load(f)
            if 'model' in d:
                merged[d['model']] = d
    except Exception:
        pass

if os.path.exists(res_file):
    try:
        with open(res_file, 'r') as f:
            d = json.load(f)
            if isinstance(d, dict): merged.update(d)
    except Exception:
        pass

if merged:
    print('='*120)
    print(f'{"GRAND MULTI-SEED BENCHMARK LEADERBOARD (SEEDS 1, 2, 3, 4, 5)":^120}')
    print('='*120)
    print(f'{"Rank":<5} | {"Model":<18} | {"Params":<8} | {"Test IoU (Mean±Std)":<22} | {"Bolivia IoU (Mean±Std)":<24} | {"Test F1 (Mean±Std)":<20} | {"Val IoU (Mean±Std)":<18}')
    print('-'*120)
    
    def get_test_iou(x):
        if 'mean' in x and 'test_iou' in x['mean']: return x['mean']['test_iou']
        return x.get('test_water_iou', 0.0)
        
    sorted_items = sorted(merged.values(), key=get_test_iou, reverse=True)
    for rank, res in enumerate(sorted_items, 1):
        m_name = res.get('model', 'unknown').upper()
        p = f"{res.get('params_M', 0):.2f}M"
        if 'mean' in res and 'std' in res:
            m = res['mean']
            s = res['std']
            test_str = f"{m['test_iou']:.4f} ± {s['test_iou']:.4f}"
            bolivia_str = f"{m['bolivia_iou']:.4f} ± {s['bolivia_iou']:.4f}"
            f1_str = f"{m['test_f1']:.4f} ± {s['test_f1']:.4f}"
            val_str = f"{m['val_iou']:.4f} ± {s['val_iou']:.4f}"
        else:
            test_str = f"{res.get('test_water_iou', 0):.4f}"
            bolivia_str = f"{res.get('bolivia_water_iou', 0):.4f}"
            f1_str = f"{res.get('test_f1', 0):.4f}"
            val_str = f"{res.get('best_val_water_iou', 0):.4f}"
        print(f"{rank:<5} | {m_name:<18} | {p:<8} | {test_str:<22} | {bolivia_str:<24} | {f1_str:<20} | {val_str:<18}")
    print('='*120)
else:
    print('No results found in', save_dir)
